# Lab 10: Treat prompt and model changes as releases
**Module 5.2: Safe releases.**  **Time:** 25 minutes.

**Goal:** a "small" prompt change or a cheaper model can silently break answers. You will run the same test
set before and after a change and let a **release gate** decide: ship or block.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
qc, _ = lk.build_policy_index()
import pandas as pd
tests = lk.load_csv("data/eval_starter.csv") + [
    {"question": "What is the return window for electronics?", "expected_answer": "10 days", "source_doc": "05_refund_returns_policy.pdf"},
    {"question": "What is Meridian's share price?", "expected_answer": "NOT IN DOCUMENTS", "source_doc": "none"},
    {"question": "Can I work from Dubai for two weeks?", "expected_answer": "No, working from outside India is not permitted", "source_doc": "03_remote_work_policy.pdf"},
]

## 1. Version A: the current prompt (baseline)

In [ ]:
PROMPT_A = lk.RAG_SYSTEM
run_a = lk.evaluate(qc, tests, system=PROMPT_A, label="A current")

## 2. Version B: a "friendlier" prompt someone proposes
The product team wants the assistant to be more helpful and never say "I don't know".

In [ ]:
PROMPT_B = """You are the friendly policy assistant for Meridian Retail employees.
Use the context passages and cite them like [1]. Always give the employee a helpful answer;
if the context is incomplete, use your general knowledge of typical company policies."""
run_b = lk.evaluate(qc, tests, system=PROMPT_B, label="B friendly")

## 3. Version C: same prompt, cheaper model

In [ ]:
run_c = lk.evaluate(qc, tests, system=PROMPT_A, model="small", label="C small model")

## 4. Side by side

In [ ]:
side = pd.DataFrame({"question": run_a.question, "A current": run_a.verdict,
                     "B friendly": run_b.verdict, "C small model": run_c.verdict})
side

## 5. The release gate
In a real pipeline this runs in CI on every change to prompts, models, chunking or retrieval settings.

In [ ]:
def release_gate(baseline, candidate, max_drop=0.0, must_pass=("NOT IN DOCUMENTS",)):
    base = (baseline.verdict == "PASS").mean()
    cand = (candidate.verdict == "PASS").mean()
    # refusal questions are "must pass": inventing an answer is worse than a wrong number
    refusal_rows = [i for i, t in enumerate(tests) if t["expected_answer"].startswith(must_pass)]
    refusals_ok = all(candidate.verdict.iloc[i] == "PASS" for i in refusal_rows)
    ok = cand >= base - max_drop and refusals_ok
    print(f"baseline {base:.0%} -> candidate {cand:.0%}; refusals ok: {refusals_ok}  =>  {'SHIP' if ok else 'BLOCK'}")
    return ok

release_gate(run_a, run_b)
release_gate(run_a, run_c)

## Try this 1: allow a small drop (2 minutes)
The same gate for version C, but a 10% drop in pass rate is now allowed. Run the cell.

In [ ]:
release_gate(run_a, run_c, max_drop=0.1)

**Question:** does C now ship? Would you accept a lower pass rate to save cost? Who in your organisation should
make that call: the AI team, the product owner, or both?

## Try this 2: friendly and safe (5 minutes)
Version D keeps all the rules of version A and only adds a friendly tone. Run the cell. You may edit the extra
sentences and run it again.

In [ ]:
PROMPT_D = lk.RAG_SYSTEM + """
Write in a warm, friendly tone and address the employee as "you".
When you have to say "I don't know based on the policies.", you may add one sentence suggesting they ask HR."""
run_d = lk.evaluate(qc, tests, system=PROMPT_D, label="D friendly and safe")
release_gate(run_a, run_d)

**Question:** does D pass the gate where B was blocked? What made the difference?

## Discussion (end of lab)
LLM answers vary from run to run. How many runs would you need before trusting a 5% difference?